In [0]:
-- League - Weekend vs Weekday Scoring
-- Average total goals by day of the week (uses dim_date).
-- Change the values in params only; everything below reads from it.
WITH params AS (                              -- change settings here only
  SELECT
    CAST(NULL AS STRING) AS team,  -- team abbreviation; CAST(NULL AS STRING) = every team
    20252026 AS season,            -- e.g. 20242025; CAST(NULL AS INT) = every season
    2 AS game_type                 -- 2 = regular season, 3 = playoffs
),
team_games AS (                          -- one row per team per finished game
  SELECT g.game_id, g.season, g.game_date, g.date_key, g.last_period_type,
         t.team_id, t.abbrev AS team, o.abbrev AS opponent,
         t.team_id = g.home_team_id AS is_home, g.home_team_id,
         COALESCE(t.abbrev = pr.team, FALSE) AS is_selected,
         IF(t.team_id = g.home_team_id, g.home_final_score, g.away_final_score) AS final_gf,
         IF(t.team_id = g.home_team_id, g.away_final_score, g.home_final_score) AS final_ga,
         CASE WHEN IF(t.team_id = g.home_team_id, g.home_final_score > g.away_final_score,
                                                  g.away_final_score > g.home_final_score) THEN 'W'
              WHEN g.game_type = 2 AND g.last_period_type IN ('OT', 'SO') THEN 'OTL'
              ELSE 'L' END AS result
  FROM nhl.gold.dim_game g
  JOIN nhl.gold.dim_team t ON t.team_id IN (g.home_team_id, g.away_team_id)
  JOIN nhl.gold.dim_team o ON o.team_id = IF(t.team_id = g.home_team_id, g.away_team_id, g.home_team_id)
  CROSS JOIN params pr
  WHERE g.is_final
    AND g.game_type = pr.game_type
    AND (pr.season IS NULL OR g.season = pr.season)
    AND (pr.team IS NULL OR t.abbrev = pr.team)
)
SELECT d.is_weekend,
       d.day_name,
       COUNT(DISTINCT tg.game_id)                              AS games,
       ROUND(AVG(tg.final_gf + tg.final_ga), 2)                AS avg_total_goals
FROM team_games tg
JOIN nhl.gold.dim_date d ON d.date_key = tg.date_key
WHERE tg.is_home OR (SELECT team FROM params) IS NOT NULL
GROUP BY d.is_weekend, d.day_name
ORDER BY avg_total_goals DESC;